In [ ]:
import numpy as np
import pandas as pd
from numba import njit, prange
import time

# ==========================================
# 1. Game Engine (Dual Pool + Wild)
# ==========================================

@njit(nogil=True, parallel=True)
def bingo_dual_pool_engine(
    n_rounds,
    cards_per_round,
    pay_table_lines
):

    MAIN_POOL_SIZE = 31
    WILD_BALL = 0

    main_pool = np.arange(MAIN_POOL_SIZE)     # 0~30 (draw pool)
    card_pool_base = np.arange(1, 31)         # 1~30 (card pool, no Wild)
    mult_pool = np.arange(1, 31)              # 1~30 (multiplier pool)

    total_cards = n_rounds * cards_per_round

    lines_arr = np.zeros(total_cards, dtype=np.int32)
    win_arr = np.zeros(total_cards, dtype=np.float64)
    draw_count_arr = np.zeros(n_rounds, dtype=np.int32)

    for i in prange(n_rounds):

        # Multiplier ball selection
        mult_draw = mult_pool.copy()
        np.random.shuffle(mult_draw)
        multiplier_balls = mult_draw[:5]

        multiplier_lookup = np.ones(MAIN_POOL_SIZE, dtype=np.float64)

        for b in multiplier_balls:
            r = np.random.random()
            if r < 0.6:
                mult = 2
            elif r < 0.82:
                mult = 3
            else:
                mult = 5
            multiplier_lookup[b] = mult

        # Base draw (9 balls)
        draw_pool = main_pool.copy()
        np.random.shuffle(draw_pool)

        total_draws = 9
        draws = draw_pool[:total_draws]

        # Wild trigger
        wild_hit = False
        for d in draws:
            if d == WILD_BALL:
                wild_hit = True
                break

        if wild_hit:
            r = np.random.random()
            if r < 0.3:
                extra = 1
            elif r < 0.7:
                extra = 2
            elif r < 0.9:
                extra = 3
            else:
                extra = 4

            total_draws += (extra + 1)

        draw_count_arr[i] = total_draws

        draws = draw_pool[:total_draws]

        drawn_lookup = np.zeros(MAIN_POOL_SIZE, dtype=np.bool_)
        for d in draws:
            drawn_lookup[d] = True

        # Card evaluation
        for c in range(cards_per_round):

            card_pool = card_pool_base.copy()
            np.random.shuffle(card_pool)
            card = card_pool[:9]

            hits = np.zeros(9, dtype=np.bool_)
            for j in range(9):
                if drawn_lookup[card[j]]:
                    hits[j] = True

            lines = 0
            involved_mask = np.zeros(9, dtype=np.bool_)

            # Rows
            if hits[0] and hits[1] and hits[2]:
                lines += 1; involved_mask[0]=1; involved_mask[1]=1; involved_mask[2]=1
            if hits[3] and hits[4] and hits[5]:
                lines += 1; involved_mask[3]=1; involved_mask[4]=1; involved_mask[5]=1
            if hits[6] and hits[7] and hits[8]:
                lines += 1; involved_mask[6]=1; involved_mask[7]=1; involved_mask[8]=1

            # Columns
            if hits[0] and hits[3] and hits[6]:
                lines += 1; involved_mask[0]=1; involved_mask[3]=1; involved_mask[6]=1
            if hits[1] and hits[4] and hits[7]:
                lines += 1; involved_mask[1]=1; involved_mask[4]=1; involved_mask[7]=1
            if hits[2] and hits[5] and hits[8]:
                lines += 1; involved_mask[2]=1; involved_mask[5]=1; involved_mask[8]=1

            total_multiplier = 1.0

            if lines > 0:
                for k in range(9):
                    if involved_mask[k]:
                        total_multiplier *= multiplier_lookup[card[k]]

            safe_lines = min(lines, len(pay_table_lines)-1)
            base_win = pay_table_lines[safe_lines]

            idx = i * cards_per_round + c
            lines_arr[idx] = lines
            win_arr[idx] = base_win * total_multiplier

    return lines_arr, win_arr, draw_count_arr


# ==========================================
# 2. Distribution Report
# ==========================================

def get_breakpoints():
    return [0.5, 1, 2, 3, 5, 10, 20, 50, 100, 500, 1000, 5000]

def generate_report(multipliers):

    s = pd.Series(multipliers)
    total_rounds = len(s)

    bins = [0] + get_breakpoints() + [float("inf")]
    cut = pd.cut(s[s > 0], bins=bins, right=True)

    grouped = s[s > 0].groupby(cut, observed=False)

    df = grouped.agg(['count', 'sum'])
    df.columns = ['Count', 'Total Return']
    df['Frequency (%)'] = df['Count'] / total_rounds * 100
    df['RTP Contribution (%)'] = df['Total Return'] / total_rounds * 100

    df = df.sort_index(ascending=False)

    zero_count = (s == 0).sum()
    zero_row = pd.DataFrame({
        'Count':[zero_count],
        'Total Return':[0],
        'Frequency (%)':[zero_count/total_rounds*100],
        'RTP Contribution (%)':[0]
    }, index=['0'])

    return pd.concat([df, zero_row])


# ==========================================
# 3. Main Simulation
# ==========================================

def main():

    N_ROUNDS = 500_000_000
    CARDS_PER_ROUND = 1

    PAY_TABLE_LINES = np.array(
        [0, 2, 5, 25, 50, 0, 250],
        dtype=np.float64
    )

    print("=== Simulation Started ===")
    start_time = time.time()

    lines_arr, win_arr, draw_counts = bingo_dual_pool_engine(
        N_ROUNDS,
        CARDS_PER_ROUND,
        PAY_TABLE_LINES
    )

    total_cards = N_ROUNDS * CARDS_PER_ROUND
    final_mults = win_arr

    line_counts = np.bincount(lines_arr, minlength=7)

    rtp = final_mults.mean() * 100
    std = np.std(final_mults)

    hit_rate = (lines_arr > 0).sum() / total_cards * 100
    hit_2plus = (lines_arr >= 2).sum() / total_cards * 100
    dead_rate = (final_mults == 0).sum() / total_cards * 100
    full_rate = line_counts[6] / total_cards * 100

    cv = std / final_mults.mean()
    avg_draws = draw_counts.mean()

    elapsed = time.time() - start_time

    print(f"\nSimulation Time: {elapsed:.2f} seconds")
    print("-"*60)

    print("Line Distribution:")
    for i in range(7):
        prob = line_counts[i] / total_cards * 100
        print(f"{i} Lines: {line_counts[i]:,} ({prob:.6f}%)")

    print("-"*60)
    print(f"RTP: {rtp:.2f}%")
    print(f"Hit Rate: {hit_rate:.4f}%")
    print(f"2+ Lines Rate: {hit_2plus:.4f}%")
    print(f"Dead Spin Rate: {dead_rate:.4f}%")
    print(f"Full Board Rate: {full_rate:.6f}%")
    print(f"Standard Deviation: {std:.4f}")
    print(f"Coefficient of Variation (CV): {cv:.4f}")
    print(f"Average Draw Count (Wild Validation): {avg_draws:.4f}")

    print("\nReturn Distribution:")
    print(generate_report(final_mults))


if __name__ == "__main__":
    main()

=== Simulation Started ===

Simulation Time: 167.02 seconds
------------------------------------------------------------
Line Distribution:
0 Lines: 426,846,565 (85.369313%)
1 Lines: 65,417,516 (13.083503%)
2 Lines: 7,189,574 (1.437915%)
3 Lines: 483,606 (0.096721%)
4 Lines: 61,234 (0.012247%)
5 Lines: 0 (0.000000%)
6 Lines: 1,505 (0.000301%)
------------------------------------------------------------
RTP: 96.11%
Hit Rate: 14.6307%
2+ Lines Rate: 1.5472%
Dead Spin Rate: 85.3693%
Full Board Rate: 0.000301%
Standard Deviation: 13.2130
Coefficient of Variation (CV): 13.7473
Average Draw Count (Wild Validation): 9.8999

Return Distribution:
